In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

df = pd.read_csv('data/heart_failure_clinical_records_dataset.csv')

df.info()

In [ ]:
target = df['DEATH_EVENT'].apply(str)
df = df.drop('DEATH_EVENT', axis=1)

scaler = StandardScaler()
df = scaler.fit_transform(df)

df

In [ ]:
from LinearRegressionClassifier import LinearRegressionClassifier
from GDClassifier import GDClassifier
from SVMClassifier import SVMClassifier

X_train, X_test, y_train, y_test = train_test_split(df, target.values, train_size=0.7)

n = len(X_train)
epochs = n * 4

model_gd = GDClassifier(
    lr=0.001,
    epochs=epochs,
    loss_function='sigmoid',
    regularization='ridge',
    beta=50
)

model_gd.fit(X_train, y_train)
prediction = model_gd.predict(X_test)

gd_score = accuracy_score(y_test, prediction)
gd_score

In [ ]:
model_reg = LinearRegressionClassifier(tau=50)

model_reg.fit(X_train, y_train)
prediction = model_reg.predict(X_test)

reg_score = accuracy_score(y_test, prediction)
reg_score

In [ ]:
model_svm = SVMClassifier(C=1.5, epochs=epochs, max_passes=3)

model_svm.fit(X_train, y_train)
prediction = model_svm.predict(X_test)

svm_score = accuracy_score(y_test, prediction)
svm_score

In [ ]:
import optuna
from sklearn.model_selection import cross_val_score

def regression_objective(trial):
    tau = trial.suggest_int('tau', 0, 250, step=5)
    score = cross_val_score(
        LinearRegressionClassifier(tau=tau), X_train, y_train, scoring='accuracy', n_jobs=10
    )
    return score.mean()

study = optuna.create_study(direction='maximize')
study.optimize(regression_objective, n_trials=50)

In [ ]:
reg_params = study.best_params
study.best_value, reg_params

In [ ]:
def gd_objective(trial):
    lr = trial.suggest_float('lr', 0.0001, 0.0101, step=0.0002)
    loss_function = trial.suggest_categorical('loss_function', ['mse', 'log', 'sigmoid'])
    regularization = trial.suggest_categorical('regularization', [None, 'lasso', 'ridge', 'elastic'])
    if regularization is None:
        alpha = 0
        beta = 0
    elif regularization == 'lasso':
        alpha = trial.suggest_int('alpha', 1, 251, step=5)
        beta = 0
    elif regularization == 'ridge':
        alpha = 0
        beta = trial.suggest_int('beta', 1, 251, step=5)
    else:
        alpha = trial.suggest_int('alpha', 1, 251, step=5)
        beta = trial.suggest_int('beta', 1, 251, step=5)
    classifier = GDClassifier(
        lr=lr,
        epochs=epochs,
        loss_function=loss_function,
        regularization=regularization,
        alpha=alpha,
        beta=beta
    )
    score = cross_val_score(
        classifier, X_train, y_train, scoring='accuracy', n_jobs=10
    )
    return score.mean()

study = optuna.create_study(direction='maximize')
study.optimize(gd_objective, n_trials=50)

In [ ]:
gd_params = study.best_params
study.best_value, gd_params

In [ ]:
def svm_objective(trial):
    C = trial.suggest_float('C', 0.1, 2.0, step=0.1)
    max_passes = trial.suggest_int('max_passes', 1, 5, step=1)
    kernel = trial.suggest_categorical('kernel', [None, 'sqr', 'sigmoid', 'radial'])
    if kernel is not None and kernel == 'radial':
        beta = trial.suggest_float('beta', 1, 101, step=2)
    else:
        beta = 0
    classifier = SVMClassifier(C=C, max_passes=max_passes, epochs=epochs, kernel=kernel, beta=beta)
    score = cross_val_score(
        classifier, X_train, y_train, scoring='accuracy', n_jobs=10
    )
    return score.mean()

study = optuna.create_study(direction='maximize')
study.optimize(svm_objective, n_trials=50, n_jobs=10)

In [ ]:
svm_params = study.best_params
study.best_value, svm_params

In [ ]:
# train loss curve

import matplotlib.pyplot as plt
import numpy as np

model_gd = GDClassifier(epochs=epochs, track_loss=True, **gd_params)
model_gd.fit(X_train, y_train)
curve_indices = np.linspace(0, model_gd.completed_epochs - 1, 100, dtype=int)
plt.plot(curve_indices, np.array(model_gd.loss_values)[curve_indices])
plt.show()

In [ ]:
model_svm = SVMClassifier(epochs=epochs, track_loss=True, **svm_params)
model_svm.fit(X_train, y_train)
plt.plot(range(model_svm.completed_epochs), model_svm.loss_values)
plt.show()

In [ ]:
# model score curve

model_reg = LinearRegressionClassifier(**reg_params)
model_reg.fit(X_train, y_train)
prediction = model_reg.predict(X_test)
reg_score = accuracy_score(y_test, prediction)

model_gd = GDClassifier(epochs=epochs, track_score=True, **gd_params)
model_gd.fit(X_train, y_train, X_test=X_test, y_test=y_test)
curve_indices = np.linspace(0, model_gd.completed_epochs - 1, 100, dtype=int)
plt.plot(curve_indices, np.array(model_gd.score_values)[curve_indices], c='b', label='gd')
plt.plot(curve_indices, list(map(lambda _: reg_score, curve_indices)), c='g', label='reg')
plt.legend()
plt.show()

In [ ]:
model_svm = SVMClassifier(epochs=epochs, track_score=True, **svm_params)
model_svm.fit(X_train, y_train, X_test=X_test, y_test=y_test)
curve_indices = np.linspace(0, model_svm.completed_epochs - 1, 100, dtype=int)
plt.plot(curve_indices, np.array(model_svm.score_values)[curve_indices], c='y', label='svm')
plt.plot(curve_indices, list(map(lambda _: reg_score, curve_indices)), c='g', label='reg')
plt.legend()
plt.show()

In [ ]:
from sklearn.linear_model import RidgeClassifier, SGDClassifier
from sklearn.svm import SVC

lib_reg = RidgeClassifier(alpha=50)
lib_reg.fit(X_train, y_train)
lib_reg.score(X_test, y_test)

In [ ]:
lib_gd = SGDClassifier(
    learning_rate='constant',
    loss='perceptron',
    eta0=0.001,
    max_iter=epochs,
    alpha=50
)
lib_gd.fit(X_train, y_train)
lib_gd.score(X_test, y_test)

In [ ]:
lib_svm = SVC(
    C=1.5,
    max_iter=epochs,
    kernel='linear'
)
lib_svm.fit(X_train, y_train)
lib_svm.score(X_test, y_test)

In [ ]:
def objective(trial):
    alpha = trial.suggest_float('alpha', 0.0, 250.0, step=5)
    score = cross_val_score(
        RidgeClassifier(alpha=alpha), X_train, y_train
    )
    return score.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

In [ ]:
lib_reg_params = study.best_params
study.best_value, lib_reg_params

In [ ]:
def objective(trial):
    eta0 = trial.suggest_float('eta0', 0.0001, 0.0101, step=0.0002)
    loss = trial.suggest_categorical('loss', ['log_loss', 'perceptron', 'squared_error'])
    penalty = trial.suggest_categorical('penalty', [None, 'l1', 'l2', 'elasticnet'])
    if penalty is not None:
        alpha = trial.suggest_int('alpha', 1, 251, step=5)
    else:
        alpha = 0
    if penalty == 'elasticnet':
        l1_ratio = trial.suggest_float('l1_ratio', 0.0, 1.0, step=0.05)
    else:
        l1_ratio = 0
    classifier = SGDClassifier(
        loss=loss,
        penalty=penalty,
        alpha=alpha,
        l1_ratio=l1_ratio,
        max_iter=epochs,
        learning_rate='constant',
        eta0=eta0
    )
    score = cross_val_score(
        classifier, X_train, y_train
    )
    return score.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

In [ ]:
lib_gd_params = study.best_params
study.best_value, lib_gd_params

In [ ]:
def objective(trial):
    C = trial.suggest_float('C', 0.1, 2.0, step=0.1)
    kernel = trial.suggest_categorical('kernel', ['linear', 'poly', 'rbf', 'sigmoid'])
    if kernel == 'poly':
        degree = trial.suggest_int('degree', 2, 10)
    else:
        degree = 0
    if kernel in ['poly', 'rbf', 'sigmoid']:
        gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])
    else:
        gamma = 0
    if kernel == 'poly' or kernel == 'sigmoid':
        coef0 = trial.suggest_int('coef0', 1, 10)
    else:
        coef0 = 0
    classifier = SVC(
        C=C,
        kernel=kernel,
        degree=degree,
        gamma=gamma,
        coef0=coef0,
        max_iter=epochs
    )
    score = cross_val_score(
        classifier, X_train, y_train
    )
    return score.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

In [ ]:
lib_svm_params = study.best_params
study.best_value, lib_svm_params

In [ ]:
lib_reg_best = RidgeClassifier(**lib_reg_params)
lib_reg_best.fit(X_train, y_train)
reg_best_score = lib_reg_best.score(X_test, y_test)

curve_indices = np.linspace(0, model_gd.completed_epochs - 1, 100, dtype=int)
plt.plot(curve_indices, np.array(model_gd.score_values)[curve_indices], c='b', label='gd')
plt.plot(curve_indices, list(map(lambda _: reg_score, curve_indices)), c='g', label='reg')
plt.plot(curve_indices, list(map(lambda _: reg_best_score, curve_indices)), c='c', label='lib_reg')
plt.legend()
plt.show()

In [ ]:
curve_indices = np.linspace(0, model_svm.completed_epochs - 1, 100, dtype=int)
plt.plot(curve_indices, np.array(model_svm.score_values)[curve_indices], c='y', label='svm')
plt.plot(curve_indices, list(map(lambda _: reg_score, curve_indices)), c='g', label='reg')
plt.plot(curve_indices, list(map(lambda _: reg_best_score, curve_indices)), c='c', label='lib_reg')
plt.legend()
plt.show()

In [ ]:
lib_gd_best = SGDClassifier(max_iter=epochs, **lib_gd_params)
lib_gd_best.fit(X_train, y_train)

model_gd.theta, lib_gd_best.coef_

In [ ]:
lib_svm_best = SVC(max_iter=epochs, **lib_svm_params)
lib_svm_best.fit(X_train, y_train)
lib_svm_best.dual_coef_